# 02 - Experiments

Feature-engineering and model comparisons, threshold analysis and feature importance.
All choices are made with a **validation split taken from the training data** so the final test set stays untouched.
The full, final pipeline is `python -m src.train`.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

from src.data import SEED, clean, engineer
from src.models import best_f1_threshold, tune_logistic_regression
from sklearn.model_selection import StratifiedKFold

X, y = engineer(clean(pd.read_csv("../data/cs-training.csv")))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)

# Work only inside the training set: carve out a validation split.
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.25, stratify=y_tr, random_state=SEED)
len(X_fit), len(X_val)

## Does feature engineering help the baseline?

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from src.data import RAW_FEATURES

def lr_auc(cols):
    pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                     ("clf", LogisticRegression(C=0.1, max_iter=3000, class_weight="balanced"))])
    pipe.fit(X_fit[cols], y_fit)
    return roc_auc_score(y_val, pipe.predict_proba(X_val[cols])[:, 1])

print("raw features only :", round(lr_auc(RAW_FEATURES), 4))
print("all 17 features   :", round(lr_auc(list(X.columns)), 4))

## XGBoost on the validation split

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=600, max_depth=5, learning_rate=0.01, subsample=0.8, colsample_bytree=0.8,
                    min_child_weight=20, gamma=0.5, reg_lambda=5, tree_method="hist", importance_type="gain",
                    eval_metric="auc", random_state=SEED, n_jobs=-1)
xgb.fit(X_fit, y_fit)
p_val = xgb.predict_proba(X_val)[:, 1]
print("validation ROC-AUC:", round(roc_auc_score(y_val, p_val), 4), "| PR-AUC:", round(average_precision_score(y_val, p_val), 4))

## Threshold analysis

F1-optimal threshold versus alternatives. A real lender would pick the threshold from the cost of a missed default versus a rejected good borrower.

In [ ]:
prec, rec, thr = precision_recall_curve(y_val, p_val)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(thr, prec[:-1], label="precision")
ax.plot(thr, rec[:-1], label="recall")
ax.axvline(best_f1_threshold(y_val, p_val), color="k", ls="--", label="F1-optimal")
ax.set_xlabel("threshold"); ax.legend(); plt.tight_layout()

## Feature importance (gain)

`total_past_due` is the sum of the three past-due columns, so their importance is shared between correlated features. Read the ranking as 'past delinquency matters most', not as exact per-feature effects.

In [ ]:
pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False).round(3).head(10)